# Experiment: Target-dependent Sentiment Analysis (TDSA)

Objective: compute sentiment toward specific targets by evaluating local context windows
around each target mention.


## 1. Setup


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
np.random.seed(42)
random.seed(42)


## 2. Configuration


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.json"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.json"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "data" / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

for d in [PROCESSED_DIR, SUBMISSION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RANDOM_SEED = 42
PROJECT_ROOT


## 3. Data Loading


In [ ]:
from sentiment_project.core import LABEL_COL, REVIEW_COL, load_test_dataframe, load_train_dataframe

train_df = load_train_dataframe(TRAIN_PATH)
test_df = load_test_dataframe(TEST_PATH)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")


## 4. Data Validation


In [ ]:
assert train_df[LABEL_COL].isin([0, 1]).all()
assert train_df[REVIEW_COL].str.len().gt(0).all()


## 5. Exploratory Analysis


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words="english", min_df=15, ngram_range=(1, 1))
X = cv.fit_transform(train_df[REVIEW_COL])
freq = np.asarray(X.sum(axis=0)).ravel()
terms = np.array(cv.get_feature_names_out())
term_df = pd.DataFrame({"term": terms, "count": freq}).sort_values("count", ascending=False)
term_df.head(30)


## 6. Feature Engineering


In [ ]:
from sentiment_project.core import build_tfidf_logreg_pipeline

model = build_tfidf_logreg_pipeline(random_state=RANDOM_SEED)
model.fit(train_df[REVIEW_COL], train_df[LABEL_COL])

targets = ["quality", "price", "size", "service", "shipping", "design", "battery", "material"]
targets


## 7. Modeling


In [ ]:
import re
def extract_window(text: str, target: str, window: int = 8) -> str | None:
    tokens = text.split()
    lower_tokens = [t.lower().strip(".,!?:;()[]{}\"'") for t in tokens]
    if target not in lower_tokens:
        return None
    idx = lower_tokens.index(target)
    left = max(0, idx - window)
    right = min(len(tokens), idx + window + 1)
    return " ".join(tokens[left:right])
rows = []
for target in targets:
    snippets = []
    for review in train_df[REVIEW_COL].tolist():
        window_text = extract_window(review, target)
        if window_text:
            snippets.append(window_text)
    if len(snippets) < 20:
        continue
    preds = model.predict(snippets)
    rows.append({
        "target": target,
        "mentions": len(snippets),
        "positive_rate": float(np.mean(preds == 1)),
        "negative_rate": float(np.mean(preds == 0)),
    })
target_df = pd.DataFrame(rows).sort_values("mentions", ascending=False)
target_df


## 8. Evaluation


In [ ]:
if not target_df.empty:
    ax = sns.barplot(data=target_df, x="target", y="positive_rate")
    ax.set_ylim(0, 1)
    ax.set_title("Target-dependent positive sentiment")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


## 9. Inference / Submission


In [ ]:
out = PROCESSED_DIR / "target_dependent_sentiment_summary.csv"
target_df.to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT))
target_df.head()


## 10. Conclusions / Next Steps

- TDSA offers finer-grained insights than whole-review sentiment.
- The current method uses heuristic windows around target words.
- A stronger approach is a target-aware transformer (target token markers + fine-tuning).
